In [3]:
import torch
from torchvision import transforms

from sklearn.metrics import f1_score, confusion_matrix, classification_report
import seaborn as sns


PARTE 2: PREPARACIÓN DATASET

In [4]:
# transformaciones para entrenamiento

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# transformaciones para validación y test
transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

PARTE 3: CARGAR EL DATASET

In [5]:
import os

# En Kaggle, los datasets se encuentran en /kaggle/input/<nombre-dataset>
# Ajusta el nombre del dataset según el que hayas subido en Kaggle
ruta = '/kaggle/input/datasets/amelianadiamaziuk/dataset-fusionado/dataset_fusionado'

# Verificar que el directorio existe
if not os.path.exists(ruta):
    print(f"Error: El directorio '{ruta}' no existe.")
    print("Asegúrate de haber añadido tu dataset en Kaggle y de que la ruta es correcta.")
    files = []
else:
    files = os.listdir(ruta)
    print(f"Total imágenes: {len(files)}'")
    print('Primeros 10: ')
    for f in files[:10]:
        print(f)


Total imágenes: 3'
Primeros 10: 
val
test
train


In [6]:
from torchvision import datasets
from torch.utils.data import DataLoader


class ImageFolder(datasets.ImageFolder):
  def __getitem__(self, index):
    try:
      return super(ImageFolder, self).__getitem__(index)
    except:
      return self.__getitem__(index + 1)

# Load pre-split datasets from subfolders
train_dataset = ImageFolder(f"{ruta}/train", transform=transform_train)
val_dataset = ImageFolder(f"{ruta}/val", transform=transform_val)
test_dataset = ImageFolder(f"{ruta}/test", transform=transform_val)

print(f"Total imágenes en train: {len(train_dataset)}")
print(f"Total imágenes en val: {len(val_dataset)}")
print(f"Total imágenes en test: {len(test_dataset)}")

classes = train_dataset.classes # Define 'classes' variable from one of the datasets
print(f"Clases: {classes}")



Total imágenes en train: 4745
Total imágenes en val: 1013
Total imágenes en test: 1027
Clases: ['cups', 'forks', 'glasses', 'knives', 'plates', 'spoons']


PARTE 4: CARGAR DATOS

In [7]:
train_loader = DataLoader(train_dataset, batch_size = 16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size = 16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size = 16, shuffle=False)

print(f"Batches en train: {len(train_loader)}")
print(f"Batches en val: {len(val_loader)}")
print(f"Batches en test: {len(test_loader)}")


Batches en train: 297
Batches en val: 64
Batches en test: 65


PARTE 5: ARQUITECTURA CNN

In [8]:
import torch.nn as nn

# ============================================================
# ARQUITECTURA: RED RESIDUAL (ResNet)
# ============================================================

class Block(nn.Module):
    def __init__(self, in_channels, out_channels, identity_downsample=None, stride=1):
        super().__init__()
        self.expansion = 4
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, kernel_size=1, stride=1, padding=0)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)
        self.relu = nn.ReLU()
        self.identity_downsample = identity_downsample

    def forward(self, x):
        identity = x
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))
        if self.identity_downsample:
            identity = self.identity_downsample(identity)
        x += identity
        return self.relu(x)


class ResNet(nn.Module):
    def __init__(self, layers, img_channels, num_classes):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(img_channels, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.max_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(layers[0], 64, stride=1)
        self.layer2 = self._make_layer(layers[1], 128, stride=2)
        self.layer3 = self._make_layer(layers[2], 256, stride=2)
        self.layer4 = self._make_layer(layers[3], 512, stride=2)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * 4, num_classes)

    def _make_layer(self, num_residual, out_channels, stride):
        identity_downsample = None
        layers = nn.ModuleList()
        if stride != 1 or self.in_channels != out_channels * 4:
            identity_downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * 4, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels * 4)
            )
        layers.append(Block(self.in_channels, out_channels, identity_downsample, stride))
        self.in_channels = out_channels * 4
        for _ in range(num_residual - 1):
            layers.append(Block(self.in_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.max_pool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avg_pool(x)
        x = x.reshape(x.shape[0], -1)
        return self.fc(x)


def resnet50(img_channels=3, num_classes=6):
    return ResNet([3, 4, 6, 3], img_channels, num_classes)

def resnet101(img_channels=3, num_classes=6):
    return ResNet([3, 4, 23, 3], img_channels, num_classes)

def resnet152(img_channels=3, num_classes=6):
    return ResNet([3, 4, 36, 3], img_channels, num_classes)

In [9]:
modelo = resnet50(img_channels=3, num_classes=6)
print(modelo)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (max_pool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Block(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1))
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
      (identity_downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1))
        (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=T

In [10]:
# vamos a probar que la red funciona con el tamaño de la imágenes

imagen_prueba = torch.randn(1, 3, 224, 224)
with torch.no_grad():
  salida = modelo(imagen_prueba)

print(imagen_prueba.shape)
print(salida.shape)
print(classes)

torch.Size([1, 3, 224, 224])
torch.Size([1, 6])
['cups', 'forks', 'glasses', 'knives', 'plates', 'spoons']


In [11]:
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.02, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super(SAM, self).__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups[0]['params'], **kwargs)
        self.rho = rho  # ← guardamos rho directamente

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = self.rho / (grad_norm + 1e-12)  # ← usamos self.rho
            for p in group['params']:
                if p.grad is None: continue
                e_w = p.grad * scale.to(p)
                p.add_(e_w)
                self.state[p]['e_w'] = e_w
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                p.sub_(self.state[p]['e_w'])
        self.base_optimizer.step()
        if zero_grad: self.zero_grad()

    def _grad_norm(self):
        shared_device = self.param_groups[0]['params'][0].device
        return torch.norm(torch.stack([
            p.grad.norm(p=2).to(shared_device)
            for group in self.param_groups
            for p in group['params']
            if p.grad is not None
        ]), p=2)



PARTE 6: ENTRENAMIENTO

In [13]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


modelo = resnet50(img_channels=3, num_classes=6).to(device)
optimizer = SAM(modelo.parameters(), torch.optim.SGD, lr=0.0001, momentum = 0.9)

criterio = nn.CrossEntropyLoss()
epochs = 20

hist_loss_train = []
hist_acc_train = [] # Nueva lista para la precisión de entrenamiento
hist_loss_val = []  # Nueva lista para la pérdida de validación
hist_acc_val = []

for epoch in range(epochs):

  # ENTRNAMIENTO

  modelo.train()
  loss_total_train = 0
  correctas_train = 0 # Para precisión de entrenamiento
  total_train = 0     # Para precisión de entrenamiento

  for imagenes, etiquetas in train_loader:
    imagenes = imagenes.to(device)
    etiquetas = etiquetas.to(device)

    salida = modelo(imagenes)
    loss = criterio(salida, etiquetas) # calcula error
    loss.backward() # backward pass
    optimizer.first_step(zero_grad=True) # actualiza pesos

    salida2 = modelo(imagenes) # forward pass
    loss2 = criterio(salida2, etiquetas) # calcula error
    loss2.backward() # backward pass
    optimizer.second_step(zero_grad=True) # actualiza pesos


    loss_total_train += loss.item()

    # Calcular precisión en entrenamiento
    _, predicciones_train = torch.max(salida, 1)
    correctas_train += (predicciones_train == etiquetas).sum().item()
    total_train += etiquetas.size(0)

  loss_medio_train = loss_total_train / len(train_loader)
  acc_train = correctas_train / total_train


  # fase de validación

  modelo.eval()
  correctas_val = 0
  total_val = 0
  loss_total_val = 0 # Para pérdida de validación

  with torch.no_grad():
    for imagenes, etiquetas in val_loader:
      imagenes = imagenes.to(device)
      etiquetas = etiquetas.to(device)
      salida = modelo(imagenes)
      loss_val = criterio(salida, etiquetas) # Calcula la pérdida de validación
      loss_total_val += loss_val.item()

      _, predicciones_val = torch.max(salida, 1)
      correctas_val += (predicciones_val == etiquetas).sum().item()
      total_val += etiquetas.size(0)

  loss_medio_val = loss_total_val / len(val_loader)
  acc_val = correctas_val / total_val

  hist_loss_train.append(loss_medio_train)
  hist_acc_train.append(acc_train) # Añadir precisión de entrenamiento
  hist_loss_val.append(loss_medio_val) # Añadir pérdida de validación
  hist_acc_val.append(acc_val)

  print(f"Epoch: {epoch+1}/{epochs}")
  print(f"Loss train: {loss_medio_train:.4f} | Acc train: {acc_train:.4f}")
  print(f"Loss val:   {loss_medio_val:.4f} | Acc val:   {acc_val:.4f}")

Epoch: 1/20
Loss train: 1.7638 | Acc train: 0.2312
Loss val:   1.7308 | Acc val:   0.2655
Epoch: 2/20
Loss train: 1.7254 | Acc train: 0.2508
Loss val:   1.7377 | Acc val:   0.2507
Epoch: 3/20
Loss train: 1.7040 | Acc train: 0.2670
Loss val:   1.6929 | Acc val:   0.2646
Epoch: 4/20
Loss train: 1.6994 | Acc train: 0.2613
Loss val:   1.7026 | Acc val:   0.2498
Epoch: 5/20
Loss train: 1.6760 | Acc train: 0.2927
Loss val:   1.6591 | Acc val:   0.3346
Epoch: 6/20
Loss train: 1.6635 | Acc train: 0.2986
Loss val:   1.6693 | Acc val:   0.3060
Epoch: 7/20
Loss train: 1.6454 | Acc train: 0.3138
Loss val:   1.6400 | Acc val:   0.3040
Epoch: 8/20
Loss train: 1.6263 | Acc train: 0.3203
Loss val:   1.6199 | Acc val:   0.3149
Epoch: 9/20
Loss train: 1.6139 | Acc train: 0.3389
Loss val:   1.6202 | Acc val:   0.3544
Epoch: 10/20
Loss train: 1.5781 | Acc train: 0.3452
Loss val:   1.5600 | Acc val:   0.3445
Epoch: 11/20
Loss train: 1.5432 | Acc train: 0.3770
Loss val:   1.5437 | Acc val:   0.3820
Epoch: 1

In [14]:

# EVALUACIÓN FINAL EN TEST SET
print("\n" + "="*60)
print("EVALUACIÓN EN TEST SET")
print("="*60 + "\n")

modelo.eval()
correctas_test = 0
total_test = 0
todas_predicciones_test = []
todas_etiquetas_test = []

with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        imagenes = imagenes.to(device)
        etiquetas = etiquetas.to(device)

        salida = modelo(imagenes)
        _, predicciones = torch.max(salida, 1)

        correctas_test += (predicciones == etiquetas).sum().item()
        total_test += etiquetas.size(0)

        todas_predicciones_test.extend(predicciones.cpu().numpy())
        todas_etiquetas_test.extend(etiquetas.cpu().numpy())

# Calcular métricas finales
acc_test = correctas_test / total_test
f1_test = f1_score(todas_etiquetas_test, todas_predicciones_test, average='macro')

print(f"Test Accuracy:     {acc_test:.4f}")
print(f"Test Macro F1-Score: {f1_test:.4f}")

# Classification Report detallado
print("\n" + "="*60)
print("CLASSIFICATION REPORT (TEST SET)")
print("="*60)
print(classification_report(todas_etiquetas_test, todas_predicciones_test,
                          target_names=classes, digits=4))

print("\n✓ Evaluación en test completada")


EVALUACIÓN EN TEST SET

Test Accuracy:     0.4713
Test Macro F1-Score: 0.4649

CLASSIFICATION REPORT (TEST SET)
              precision    recall  f1-score   support

        cups     0.6087    0.5158    0.5584       190
       forks     0.3956    0.2353    0.2951       153
     glasses     0.3754    0.6358    0.4721       173
      knives     0.3563    0.3690    0.3626       168
      plates     0.7538    0.5444    0.6323       180
      spoons     0.4494    0.4908    0.4692       163

    accuracy                         0.4713      1027
   macro avg     0.4899    0.4652    0.4649      1027
weighted avg     0.4965    0.4713    0.4714      1027


✓ Evaluación en test completada


### PARTE 7: Visualización de los Resultados

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, epochs +1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# Gráfico de Pérdida (Train y Val)
ax1.plot(epochs_range, hist_loss_train, 'b-o', label='Training Loss')
ax1.plot(epochs_range, hist_loss_val, 'r-o', label='Validation Loss')
ax1.set_title("Pérdida durante el entrenamiento y validación")
ax1.set_xlabel("Épocas")
ax1.set_ylabel("Pérdida")
ax1.legend()

# Gráfico de Precisión (Train y Val)
ax2.plot(epochs_range, [a * 100 for a in hist_acc_train], 'b-o', label='Training Accuracy')
ax2.plot(epochs_range, [a * 100 for a in hist_acc_val], 'r-o', label='Validation Accuracy')
ax2.set_title("Precisión durante el entrenamiento y validación")
ax2.set_xlabel("Épocas")
ax2.set_ylabel("Precisión (%)")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# FIGURA 2: CONFUSION MATRIX
# ============================================================================

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(todas_etiquetas_test, todas_predicciones_test)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes,
            cbar_kws={'label': 'Count'}, annot_kws={'size': 12})
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Confusion Matrix guardada como 'confusion_matrix.png'")

# ── 2. IMÁGENES MAL CLASIFICADAS ─────────────────────────────────
# Buscamos índices donde la predicción fue incorrecta
errores = [(i, p, e) for i, (p, e) in enumerate(zip(todas_predicciones_test, todas_etiquetas_test)) if p != e]
print(f"\nTotal imágenes mal clasificadas: {len(errores)}")

# Mostramos las primeras 9
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
axes = axes.flatten()

for idx, (i, pred, real) in enumerate(errores[:9]):
    imagen, _ = test_dataset[i]
    # Desnormalizamos para visualizar
    mean = torch.tensor([0.485, 0.456, 0.406])
    std  = torch.tensor([0.229, 0.224, 0.225])
    imagen = imagen * std[:, None, None] + mean[:, None, None]
    imagen = imagen.permute(1, 2, 0).numpy()
    imagen = np.clip(imagen, 0, 1)

    axes[idx].imshow(imagen)
    axes[idx].set_title(f'Real: {classes[real]}\nPred: {classes[pred]}', fontsize=10)
    axes[idx].axis('off')

# Ocultamos los ejes sobrantes si hay menos de 9 errores
for idx in range(len(errores[:9]), 9):
    axes[idx].axis('off')

plt.suptitle('Imágenes mal clasificadas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('misclassified.png', dpi=150)
plt.show()

In [ ]:
# ============================================================================
# RESUMEN FINAL DE EXPERIMENTO 1
# ============================================================================

print("\n" + "="*70)
print("RESUMEN FINAL - EXPERIMENTO 3 - ResNet50 + SAM")
print("="*70)

print(f"\n{'MÉTRICA':<30} {'VALIDACIÓN':<15} {'TEST':<15}")
print("-" * 60)
print(f"{'Accuracy':<30} {hist_acc_val[-1]:<15.4f} {acc_test:<15.4f}")
print(f"{'Macro F1-Score':<30}{"":<17}{f1_test:<15.4f}")
print(f"{'Training Loss (final)':<30} {hist_loss_train[-1]:<15.4f}")

print("\n" + "="*70)
print("ARCHIVOS GENERADOS")
print("="*70)
print("✓ training_curves.png")
print("✓ confusion_matrix.png")
print("✓ misclassified.png")
print("\n✓ Experimento 2 base COMPLETADO")